# Loading Data

In [375]:
import pandas as pd
import numpy as np
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)


Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/data-dump/versions/1


In [376]:
df = pd.read_csv(path + '/ganjoo_ipl_mbm_bat_final.csv')

In [377]:
df['date'] = pd.to_datetime(df['date'])

df['boundaries'] = df['fours'] + df['sixes']

df.drop(columns = ['fours', 'sixes', 'sr', 'bowler_id', 'bowler'], inplace=True)

# Career Stats

In [378]:
df['isOut'] = df['isOut'].replace('DNB', 'False')
df['isOut'] = df['isOut'].astype(str).map({'True': 1, 'False': 0}).fillna(0).astype(int)

In [379]:
df.columns

Index(['date', 'match_id', 'bat_team', 'bowl_team', 'inns', 'batter_id',
       'batter', 'runs', 'balls', 'isOut', 'wicketType', 'venue',
       'boundaries'],
      dtype='object')

In [380]:
# Sort data by batter and match date
df.sort_values(by=['batter_id', 'date'], inplace=True)

# Compute cumulative stats for each batter
df['cumulative_runs'] = df.groupby('batter')['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = df.groupby('batter')['isOut'].cumsum() - df['isOut']
df['cumulative_matches'] = df.groupby('batter').cumcount()
df['cumulative_balls'] = df.groupby('batter')['balls'].cumsum() - df['balls']
df['cumulative_boundaries'] = df.groupby('batter')['boundaries'].cumsum() - df['boundaries']

# Compute batting statistics
df['career_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['career_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['career_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['career_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['career_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
career_cols = ['career_avg', 'career_bfpm', 'career_sr', 'career_rpm', 'career_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in career_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter']).head(1).index, career_cols] = pd.NA

In [381]:
df[df['batter_id'] == 446507].tail(10)

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,wicketType,venue,boundaries,career_avg,career_rpm,career_sr,career_bpm,career_bfpm
13159,2024-04-11,1426263,Mumbai Indians,Royal Challengers Bengaluru,2,446507,Suryakumar Yadav,52,19,1,caught,Wankhede Stadium,9,33.172043,25.081301,143.621974,3.577236,17.463415
13260,2024-04-14,1426267,Mumbai Indians,Chennai Super Kings,2,446507,Suryakumar Yadav,0,2,1,caught,Wankhede Stadium,0,33.372340,25.298387,144.762344,3.620968,17.475806
13339,2024-04-18,1426271,Mumbai Indians,Punjab Kings,1,446507,Suryakumar Yadav,78,53,1,caught,Maharaja Yadavindra Singh International Cricke...,10,33.021053,25.096000,144.628861,3.592000,17.352000
13457,2024-04-22,1426276,Mumbai Indians,Rajasthan Royals,1,446507,Suryakumar Yadav,10,8,1,caught,Sawai Mansingh Stadium,2,33.489583,25.515873,144.689469,3.642857,17.634921
13592,2024-04-27,1426281,Mumbai Indians,Delhi Capitals,2,446507,Suryakumar Yadav,26,13,1,caught,Arun Jaitley Stadium,5,33.247423,25.393701,144.618834,3.629921,17.559055
13697,2024-04-30,1426286,Mumbai Indians,Lucknow Super Giants,1,446507,Suryakumar Yadav,10,6,1,caught,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,1,33.173469,25.398438,144.939813,3.640625,17.523438
13782,2024-05-03,1426289,Mumbai Indians,Kolkata Knight Riders,2,446507,Suryakumar Yadav,56,35,1,caught,Wankhede Stadium,8,32.939394,25.279070,144.997777,3.620155,17.434109
13879,2024-05-06,1426293,Mumbai Indians,Sunrisers Hyderabad,2,446507,Suryakumar Yadav,102,51,0,not out,Wankhede Stadium,18,33.170000,25.515385,145.227671,3.653846,17.569231
14001,2024-05-11,1426298,Mumbai Indians,Kolkata Knight Riders,2,446507,Suryakumar Yadav,11,14,1,caught,Eden Gardens,1,34.190000,26.099237,146.423983,3.763359,17.824427
14118,2024-05-17,1426305,Mumbai Indians,Lucknow Super Giants,2,446507,Suryakumar Yadav,0,3,1,caught,Wankhede Stadium,0,33.960396,25.984848,146.019583,3.742424,17.795455


# Innings Stats

In [382]:
# Sort data by batter and match date
df.sort_values(by=['batter', 'date'], inplace=True)

# Compute cumulative stats for each batter
df['cumulative_runs'] = df.groupby(['batter', 'inns'])['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = df.groupby(['batter', 'inns'])['isOut'].cumsum() - df['isOut']
df['cumulative_matches'] = df.groupby(['batter', 'inns']).cumcount()
df['cumulative_balls'] = df.groupby(['batter', 'inns'])['balls'].cumsum() - df['balls']
df['cumulative_boundaries'] = df.groupby(['batter', 'inns'])['boundaries'].cumsum() - df['boundaries']

# Compute batting statistics
df['inns_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['inns_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['inns_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['inns_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['inns_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
inns_cols = ['inns_avg', 'inns_bfpm', 'inns_sr', 'inns_rpm', 'inns_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in inns_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter', 'inns']).head(1).index, inns_cols] = pd.NA

In [383]:
df[(df['batter_id'] == 328026) & (df['inns'] == 1)]

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,career_avg,career_rpm,career_sr,career_bpm,career_bfpm,inns_avg,inns_rpm,inns_sr,inns_bpm,inns_bfpm
7762,2020-09-28,1216547,Royal Challengers Bangalore,Mumbai Indians,1,328026,Isuru Udana,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7279,2020-10-10,1216525,Royal Challengers Bangalore,Chennai Super Kings,1,328026,Isuru Udana,0,0,0,...,1.0,0.333333,33.333333,0.000000,1.00,0.0,0.0,0.000000,0.00,0.00
7606,2020-10-12,1216540,Royal Challengers Bangalore,Kolkata Knight Riders,1,328026,Isuru Udana,0,0,0,...,1.0,0.250000,33.333333,0.000000,0.75,0.0,0.0,0.000000,0.00,0.00
7405,2020-10-15,1216531,Royal Challengers Bangalore,Punjab Kings,1,328026,Isuru Udana,10,5,0,...,1.0,0.200000,33.333333,0.000000,0.60,0.0,0.0,0.000000,0.00,0.00
6773,2020-10-31,1216502,Royal Challengers Bangalore,Sunrisers Hyderabad,1,328026,Isuru Udana,0,1,1,...,11.0,1.375000,137.500000,0.125000,1.00,10.0,2.5,200.000000,0.25,1.25
6838,2020-11-02,1216505,Royal Challengers Bangalore,Delhi Capitals,1,328026,Isuru Udana,4,2,1,...,5.5,1.222222,122.222222,0.111111,1.00,10.0,2.0,166.666667,0.20,1.20


# Venue Stats

In [384]:
# Group by batter and venue to compute cumulative statistics
grouped = df.groupby(['batter', 'venue'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = grouped['isOut'].cumsum() - df['isOut']
df['cumulative_boundaries'] = grouped['boundaries'].cumsum() - df['boundaries']
df['cumulative_balls'] = grouped['balls'].cumsum() - df['balls']
df['cumulative_matches'] = grouped.cumcount()

# Compute batting statistics
df['ven_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['ven_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['ven_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['ven_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['ven_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
venue_cols = ['ven_avg', 'ven_bfpm', 'ven_sr', 'ven_rpm', 'ven_bpm']

# making the first val as Null
df.loc[df.groupby(['batter', 'venue']).head(1).index, venue_cols] = pd.NA

In [385]:
df[(df['batter_id'] == 714451)].head(10)

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,inns_avg,inns_rpm,inns_sr,inns_bpm,inns_bfpm,ven_avg,ven_rpm,ven_sr,ven_bpm,ven_bfpm
5229,2019-03-23,1175356,Royal Challengers Bangalore,Chennai Super Kings,1,714451,Shivam Dube,2,5,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5373,2019-03-28,1175362,Royal Challengers Bangalore,Mumbai Indians,2,714451,Shivam Dube,9,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5465,2019-03-31,1175366,Royal Challengers Bangalore,Sunrisers Hyderabad,2,714451,Shivam Dube,5,7,1,...,9.000000,9.000000,180.000000,1.000000,5.000000,NaN,NaN,NaN,NaN,NaN
6233,2019-04-28,1178421,Royal Challengers Bangalore,Delhi Capitals,2,714451,Shivam Dube,24,16,1,...,14.000000,7.000000,116.666667,1.000000,6.000000,NaN,NaN,NaN,NaN,NaN
7473,2020-09-21,1216534,Royal Challengers Bangalore,Sunrisers Hyderabad,1,714451,Shivam Dube,7,8,1,...,2.000000,2.000000,40.000000,0.000000,5.000000,NaN,NaN,NaN,NaN,NaN
6953,2020-09-24,1216510,Royal Challengers Bangalore,Punjab Kings,2,714451,Shivam Dube,12,12,1,...,19.000000,12.666667,135.714286,1.333333,9.333333,7.0,7.000000,87.500000,0.000000,8.0
7757,2020-09-28,1216547,Royal Challengers Bangalore,Mumbai Indians,1,714451,Shivam Dube,27,10,0,...,4.500000,4.500000,69.230769,0.000000,6.500000,9.5,9.500000,95.000000,0.500000,10.0
7048,2020-10-03,1216514,Royal Challengers Bangalore,Rajasthan Royals,2,714451,Shivam Dube,0,0,0,...,16.666667,12.500000,125.000000,1.250000,10.000000,NaN,NaN,NaN,NaN,NaN
7154,2020-10-05,1216519,Royal Challengers Bangalore,Delhi Capitals,2,714451,Shivam Dube,11,12,1,...,16.666667,10.000000,125.000000,1.000000,8.000000,23.0,15.333333,153.333333,1.666667,10.0
7274,2020-10-10,1216525,Royal Challengers Bangalore,Chennai Super Kings,1,714451,Shivam Dube,22,14,0,...,18.000000,12.000000,156.521739,1.333333,7.666667,19.0,14.250000,135.714286,1.500000,10.5


# Season Stats

In [386]:
df['season'] = df['date'].dt.year

# Group by batter and venue to compute cumulative statistics
grouped = df.groupby(['batter', 'season'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_dismissals'] = grouped['isOut'].cumsum() - df['isOut']
df['cumulative_boundaries'] = grouped['boundaries'].cumsum() - df['boundaries']
df['cumulative_balls'] = grouped['balls'].cumsum() - df['balls']
df['cumulative_matches'] = grouped.cumcount()

# Compute batting statistics
df['ssn_avg'] = df['cumulative_runs'] / df['cumulative_dismissals'].replace(0, 1)
df['ssn_rpm'] = df['cumulative_runs'] / df['cumulative_matches']
df['ssn_sr'] = (df['cumulative_runs'] * 100) / df['cumulative_balls'].replace(0, 1)
df['ssn_bpm'] = df['cumulative_boundaries'] / df['cumulative_matches']
df['ssn_bfpm'] = df['cumulative_balls'] / df['cumulative_matches']

# Drop temporary columns
df.drop(columns=['cumulative_runs',	'cumulative_dismissals',	'cumulative_balls',	'cumulative_matches',	'cumulative_boundaries'], inplace=True)

# Columns to process
ssn_cols = ['ssn_avg', 'ssn_bfpm', 'ssn_sr', 'ssn_rpm', 'ssn_bpm']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in ssn_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['batter', 'season']).head(1).index, ssn_cols] = pd.NA

In [387]:
df[(df['batter_id'] == 423838) & (df['season'] == 2017)]

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,isOut,...,ven_rpm,ven_sr,ven_bpm,ven_bfpm,season,ssn_avg,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm
3648,2017-05-09,1082639,Punjab Kings,Kolkata Knight Riders,1,423838,Rahul Tewatia,15,8,0,...,NaN,NaN,NaN,NaN,2017,NaN,NaN,NaN,NaN,NaN
3694,2017-05-11,1082641,Punjab Kings,Mumbai Indians,1,423838,Rahul Tewatia,0,0,0,...,NaN,NaN,NaN,NaN,2017,15.0,15.0,187.5,3.0,8.0
3779,2017-05-14,1082645,Punjab Kings,Rising Pune Supergiant,1,423838,Rahul Tewatia,4,3,1,...,NaN,NaN,NaN,NaN,2017,15.0,7.5,187.5,1.5,4.0


# Last5 Match Stats

In [388]:
# Sort batting data by batter and match date
df.sort_values(by=['batter', 'date'], inplace=True)

# ------------------------------------------------------------------------------
# 1) batter Last 5 Match Rolling batter's Average
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('batter')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_wickets = df.groupby('batter')['isOut'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
).replace(0,1)

df['last5_avg'] = rolling_runs / rolling_wickets

# ------------------------------------------------------------------------------
# 2) batter Last 5 Match Rolling Strike Rate
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('batter')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_overs = df.groupby('batter')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
).replace(0,1
          )
df['last5_eco'] = rolling_runs / rolling_overs

# ------------------------------------------------------------------------------
# 3) batter Last 5 Match Rolling runs per match
# ------------------------------------------------------------------------------
df['last5_rpm'] = df.groupby('batter')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 4) batter Last 5 Match Rolling ball faced per Match
# ------------------------------------------------------------------------------
df['last5_bfpm'] = df.groupby('batter')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 5) batter Last 5 Match Rolling boundaries faced per Match
# ------------------------------------------------------------------------------
df['last5_bpm'] = df.groupby('batter')['boundaries'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# Cleaning and Exporting

In [389]:
df.drop(columns = ['runs', 'balls', 'isOut', 'wicketType', 'boundaries', 'season'], inplace=True)

In [390]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14268 entries, 100 to 3812
Data columns (total 33 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         14268 non-null  datetime64[ns]
 1   match_id     14268 non-null  int64         
 2   bat_team     14268 non-null  object        
 3   bowl_team    14268 non-null  object        
 4   inns         14268 non-null  int64         
 5   batter_id    14268 non-null  int64         
 6   batter       14268 non-null  object        
 7   venue        14268 non-null  object        
 8   career_avg   13795 non-null  float64       
 9   career_rpm   13795 non-null  float64       
 10  career_sr    13795 non-null  float64       
 11  career_bpm   13795 non-null  float64       
 12  career_bfpm  13795 non-null  float64       
 13  inns_avg     13385 non-null  float64       
 14  inns_rpm     13385 non-null  float64       
 15  inns_sr      13385 non-null  float64       
 16  inns_bpm